# AD Variant Analysis – Example Notebook

This notebook walks through analyzing de novo single nucleotide variants (SNVs) in transcription factor (TF) protein domains using the `ad-variant-analysis` package.

## Installation

Install the package from the repository root:

```bash
pip install .
```

The package also requires **bedtools** for genomic interval operations:

```bash
# Ubuntu / Debian
sudo apt-get install bedtools

# macOS (Homebrew)
brew install bedtools

# Conda
conda install -c bioconda bedtools
```

In [ ]:
import pandas as pd
from pathlib import Path

# Package-level import to confirm installation
import AD_variant_analysis
print(f"ad-variant-analysis version: {AD_variant_analysis.__version__}")

## 1. Input data

### 1a. TF domain annotation table

A tab-separated file with one row per transcript, containing:
- UniProt and Ensembl IDs
- Amino acid coordinate ranges for each domain type (DBD, AD, RD, Bif, IDR)

Coordinates are 1-based and formatted as comma-separated `start-end` ranges, e.g. `10-50,60-90`.

In [ ]:
all_TFs = pd.read_csv("all_TFs_table_proteins_with_IDR.txt", sep="\t", index_col=0)
print(f"{len(all_TFs):,} transcription factors loaded")
all_TFs.head()

In [ ]:
# Summary: how many TFs have each domain annotated?
domain_cols = ["DBD_coords", "AD_coords", "RD_coords", "Bif_coords", "IDR_coords"]
domain_counts = all_TFs[domain_cols].notna().sum()
domain_counts.index = [c.replace("_coords", "") for c in domain_cols]
domain_counts.rename("TFs with annotation").to_frame()

### 1b. Variants BED file

A 6-column BED file (no header) with de novo SNVs:

| col | description |
|-----|-------------|
| 0   | chromosome  |
| 1   | start (0-based) |
| 2   | end         |
| 3   | reference allele |
| 4   | alternate allele |
| 5   | score / family ID |

In [ ]:
variants = pd.read_csv(
    "SFARI_TF_de_novo_variants_gpf.bed",
    sep="\t",
    header=None,
    names=["chrom", "start", "end", "ref", "alt", "score"],
)
print(f"{len(variants):,} variants loaded")
variants.head()

In [ ]:
# Variant distribution by chromosome
variants["chrom"].value_counts().sort_index()

## 2. Pipeline

The full analysis runs as a series of CLI commands (or equivalent Python API calls).

```
CDS BED files  +  Variants BED
        |                |
        v                v
  intersect-variants          <- step 1: find variants in coding regions
        |
        v
  classify-snvs               <- step 2: Syn / Non-Syn / Nonsense
        |
        v
TF domain annotation --> classify-domain-snvs   <- step 3: per-domain results
```

### Prerequisites

Steps 1-3 require **CDS BED files** (one per Ensembl transcript), available from
[Ensembl BioMart](https://www.ensembl.org/biomart/martview/).
Steps 2-3 additionally require **protein** and **DNA transcript** reference
sequences stored as Python pickle files (dict of `ENST_ID -> sequence`).

Set the paths below to match your local data:

In [ ]:
# --- adjust these paths to your data ---
CDS_DIR             = Path("cds_beds/")                        # CDS BED files (one per ENST)
VARIANTS_BED        = Path("SFARI_TF_de_novo_variants_gpf.bed")
PROTEINS_PKL        = Path("proteins.pkl")
DNA_TRANSCRIPTS_PKL = Path("dna_transcripts.pkl")
TF_TABLE            = Path("all_TFs_table_proteins_with_IDR.txt")
DOMAIN_BEDS_DIR     = Path("domain_beds/")                     # genomic domain BED files (from classify-domain-snvs)
OUTPUT_DIR          = Path("output/")

### Step 1 – Intersect variants with CDS regions

Finds which variants overlap coding sequences. Sorted files are cached in
`output/sorted/` and reused on subsequent runs.

In [ ]:
! intersect-variants \
    --cds-dir    {CDS_DIR} \
    --variants   {VARIANTS_BED} \
    --output-dir {OUTPUT_DIR}

### Step 2 – Classify CDS variants

Labels each CDS variant as:
- **Syn** – synonymous (no amino acid change)
- **Non-Syn** – non-synonymous (amino acid change)
- **Nonsense** – introduces a stop codon

In [ ]:
! classify-snvs \
    --input-dir       {OUTPUT_DIR}/intersections \
    --output-dir      {OUTPUT_DIR}/classified \
    --proteins        {PROTEINS_PKL} \
    --dna-transcripts {DNA_TRANSCRIPTS_PKL} \
    --sorted-cds-dir  {OUTPUT_DIR}/sorted/cds

### Step 3 – Classify variants by protein domain

Intersects classified SNVs with each TF's domain regions (DBD, AD, RD, Bif, IDR)
to identify which variants fall within functional domains and their effect.

Domain BED files (genomic coordinates) can be generated from the TF annotation
table and CDS BED files using `intersect-domain-variants`.

In [ ]:
! classify-domain-snvs \
    --domain-dir         {DOMAIN_BEDS_DIR} \
    --classified-snv-dir {OUTPUT_DIR}/classified \
    --output-dir         {OUTPUT_DIR}/domain_snvs \
    --mapping            {TF_TABLE}

## 3. Exploring results

After the pipeline completes, the per-domain output files can be loaded into
pandas for downstream analysis.

In [ ]:
result_files = list((OUTPUT_DIR / "domain_snvs").glob("*.bed"))
non_empty = [f for f in result_files if f.stat().st_size > 0]
print(f"{len(result_files)} total output files, {len(non_empty)} with domain variants")

# Load and concatenate all non-empty results
col_names = [
    "dom_chrom", "dom_start", "dom_end", "domain_type", "ensg", "score", "strand", "enst",
    "var_chrom", "var_start", "var_end", "enst2", "strand2", "mut_pos", "ref_nt", "alt_nt",
    "wt_aa", "mt_aa", "classification",
]

if non_empty:
    dfs = []
    for f in non_empty:
        df = pd.read_csv(f, sep="\t", header=None)
        df.columns = col_names[: len(df.columns)]
        dfs.append(df)
    results = pd.concat(dfs, ignore_index=True)
    print(f"{len(results):,} domain variants total")
    results.head()

In [ ]:
if non_empty:
    # Count by domain type and classification
    summary = (
        results.groupby(["domain_type", "classification"])
        .size()
        .unstack(fill_value=0)
        .rename_axis(None, axis=1)
    )
    print("Variant counts by domain and classification:")
    summary